# Desafío Regresión 2 — Desarrollo

Este notebook desarrolla el desafío usando regresión lineal con `sklearn`, separación entrenamiento-validación, métricas de desempeño y selección simple de atributos mediante correlación.

**Base utilizada:** `boston.csv`  
**Variable objetivo:** `medv`

In [1]:
%matplotlib inline

import warnings
warnings.filterwarnings("ignore")

from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn import linear_model, datasets
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_squared_error, r2_score

sns.set_theme(style="whitegrid")
plt.rcParams["figure.figsize"] = (9, 4)

## Ejercicio 1: Preparar el ambiente de trabajo

Se importan las librerías, se carga `boston.csv`, se elimina `Unnamed: 0` y se obtienen medidas descriptivas.

In [2]:
boston_path = Path("boston.csv")
if not boston_path.exists():
    boston_path = Path("/mnt/data/boston.csv")

boston = pd.read_csv(boston_path)
boston = boston.drop(columns=["Unnamed: 0"])

boston.head()

,crim,zn,indus,chas,nox,rm,age,dis,rad,tax,ptratio,black,lstat,medv
0,0.00632,18.0,2.31,0,0.538,6.575,65.2,4.0900,1,296,15.3,396.90,4.98,24.0
1,0.02731,0.0,7.07,0,0.469,6.421,78.9,4.9671,2,242,17.8,396.90,9.14,21.6
2,0.02729,0.0,7.07,0,0.469,7.185,61.1,4.9671,2,242,17.8,392.83,4.03,34.7
3,0.03237,0.0,2.18,0,0.458,6.998,45.8,6.0622,3,222,18.7,394.63,2.94,33.4
4,0.06905,0.0,2.18,0,0.458,7.147,54.2,6.0622,3,222,18.7,396.90,5.33,36.2


In [3]:
boston.describe().round(3)

,crim,zn,indus,chas,nox,rm,age,dis,rad,tax,ptratio,black,lstat,medv
count,506.000,506.000,506.000,506.000,506.000,506.000,506.000,506.000,506.000,506.000,506.000,506.000,506.000,506.000
mean,3.614,11.364,11.137,0.069,0.555,6.285,68.575,3.795,9.549,408.237,18.456,356.674,12.653,22.533
std,8.602,23.322,6.860,0.254,0.116,0.703,28.149,2.106,8.707,168.537,2.165,91.295,7.141,9.197
min,0.006,0.000,0.460,0.000,0.385,3.561,2.900,1.130,1.000,187.000,12.600,0.320,1.730,5.000
25%,0.082,0.000,5.190,0.000,0.449,5.885,45.025,2.100,4.000,279.000,17.400,375.378,6.950,17.025
50%,0.257,0.000,9.690,0.000,0.538,6.208,77.500,3.207,5.000,330.000,19.050,391.440,11.360,21.200
75%,3.677,12.500,18.100,0.000,0.624,6.624,94.075,5.188,24.000,666.000,20.200,396.225,16.955,25.000
max,88.976,100.000,27.740,1.000,0.871,8.780,100.000,12.126,24.000,711.000,22.000,396.900,37.970,50.000


## Ejercicio 2: División de la muestra

Se separa la matriz de atributos `X` y el vector objetivo `y`. Luego se generan conjuntos de entrenamiento y validación con `test_size=0.33` y una semilla pseudoaleatoria.

In [4]:
X = boston.drop(columns=["medv"])
y = boston["medv"]

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.33, random_state=11238
)

print("Tamaño X_train:", X_train.shape)
print("Tamaño X_test:", X_test.shape)
print("Tamaño y_train:", y_train.shape)
print("Tamaño y_test:", y_test.shape)

Tamaño X_train: (339, 13)
Tamaño X_test: (167, 13)
Tamaño y_train: (339,)
Tamaño y_test: (167,)


## Ejercicio 3: Generación de modelos

Se implementan dos versiones de regresión lineal:

1. Modelo con intercepto.
2. Modelo sin intercepto.

In [5]:
model_intercept = linear_model.LinearRegression(fit_intercept=True)
model_no_intercept = linear_model.LinearRegression(fit_intercept=False)

model_intercept.fit(X_train, y_train)
model_no_intercept.fit(X_train, y_train)

yhat_intercept = model_intercept.predict(X_test)
yhat_no_intercept = model_no_intercept.predict(X_test)

print("Intercepto modelo con intercepto:", round(model_intercept.intercept_, 4))
print("Intercepto modelo sin intercepto:", round(model_no_intercept.intercept_, 4))

Intercepto modelo con intercepto: 30.3285
Intercepto modelo sin intercepto: 0.0


## Ejercicio 4: Obtención de métricas

Se crea la función `report_scores`, que imprime Error Cuadrático Promedio y `R²`.

In [6]:
def report_scores(y_pred, y_true):
    mse = mean_squared_error(y_true, y_pred)
    r2 = r2_score(y_true, y_pred)

    print(f"Mean Squared Error: {mse:.4f}")
    print(f"R2: {r2:.4f}")

    return {"MSE": mse, "R2": r2}

print("Modelo con intercepto")
scores_intercept = report_scores(yhat_intercept, y_test)

print("\nModelo sin intercepto")
scores_no_intercept = report_scores(yhat_no_intercept, y_test)

Modelo con intercepto
Mean Squared Error: 30.6978
R2: 0.6005

Modelo sin intercepto
Mean Squared Error: 34.2694
R2: 0.5540


In [7]:
comparison_models = pd.DataFrame([
    {"modelo": "Con intercepto", **scores_intercept},
    {"modelo": "Sin intercepto", **scores_no_intercept}
])

comparison_models.round(4)

,modelo,MSE,R2
0,Con intercepto,30.6978,0.6005
1,Sin intercepto,34.2694,0.5540


**Selección del mejor modelo**

El modelo con intercepto presenta menor `MSE` y mayor `R²`, por lo que es el mejor entre las dos versiones evaluadas.

## Ejercicio 5: Refactorización del modelo

Se genera una función `fetch_features` que calcula la correlación entre cada atributo y la variable objetivo `medv`.

In [8]:
def fetch_features(dataframe, target="medv"):
    attr_name = []
    pearson_r = []
    abs_pearson_r = []

    for col in dataframe.columns:
        if col != target:
            corr = dataframe[col].corr(dataframe[target])
            attr_name.append(col)
            pearson_r.append(corr)
            abs_pearson_r.append(abs(corr))

    features = pd.DataFrame({
        "attribute": attr_name,
        "corr": pearson_r,
        "abs_corr": abs_pearson_r
    })

    return features.sort_values("abs_corr", ascending=False).reset_index(drop=True)

features = fetch_features(boston)
features.round(4)

,attribute,corr,abs_corr
0,lstat,-0.7377,0.7377
1,rm,0.6954,0.6954
2,ptratio,-0.5078,0.5078
3,indus,-0.4837,0.4837
4,tax,-0.4685,0.4685
5,nox,-0.4273,0.4273
6,crim,-0.3883,0.3883
7,rad,-0.3816,0.3816
8,age,-0.3770,0.3770
9,zn,0.3604,0.3604


In [9]:
top6 = features.head(6)["attribute"].tolist()
top6

['lstat', 'rm', 'ptratio', 'indus', 'tax', 'nox']

**Comentario**

Los 6 atributos con mayor correlación absoluta con `medv` son:

1. `lstat`
2. `rm`
3. `ptratio`
4. `indus`
5. `tax`
6. `nox`

`lstat`, `ptratio`, `indus`, `tax` y `nox` presentan correlación negativa con `medv`, mientras que `rm` presenta correlación positiva.

## Ejercicio 6: Refactorización del modelo predictivo

Se entrena un nuevo modelo usando sólo los 6 atributos identificados. Como en el ejercicio anterior ganó el modelo con intercepto, se mantiene `fit_intercept=True`.

In [10]:
X_refactored = boston[top6]
y = boston["medv"]

X_train_ref, X_test_ref, y_train_ref, y_test_ref = train_test_split(
    X_refactored, y, test_size=0.33, random_state=11238
)

model_refactored = linear_model.LinearRegression(fit_intercept=True)
model_refactored.fit(X_train_ref, y_train_ref)

yhat_refactored = model_refactored.predict(X_test_ref)

print("Modelo refactorizado con 6 atributos")
scores_refactored = report_scores(yhat_refactored, y_test_ref)

Modelo refactorizado con 6 atributos
Mean Squared Error: 37.5192
R2: 0.5118


In [11]:
comparison_all = pd.DataFrame([
    {"modelo": "Completo con intercepto", **scores_intercept},
    {"modelo": "Completo sin intercepto", **scores_no_intercept},
    {"modelo": "Refactorizado 6 atributos", **scores_refactored}
])

comparison_all.round(4)

,modelo,MSE,R2
0,Completo con intercepto,30.6978,0.6005
1,Completo sin intercepto,34.2694,0.5540
2,Refactorizado 6 atributos,37.5192,0.5118


**Comentario**

El modelo refactorizado usa menos variables, por lo que es más simple e interpretable. Sin embargo, en esta división de entrenamiento-validación pierde desempeño respecto del modelo completo con intercepto, porque su `MSE` aumenta y su `R²` disminuye.

## Ejercicio 7: Predicción de casos

Los arreglos entregados en el enunciado se interpretan en el siguiente orden:

`lstat`, `ptratio`, `rm`, `indus`, `tax`, `nox`

Como el modelo fue entrenado con el orden de columnas `top6`, se convierten los arrays a `DataFrame` y luego se reordenan las columnas antes de predecir.

In [12]:
array_feature_order = ["lstat", "ptratio", "rm", "indus", "tax", "nox"]

worst_neighbor = np.array([37.9, 12.6, 3.5, 27.7, 187, 0.87]).reshape(1, -1)
best_neighbor = np.array([1.73, 22, 8.7, 0.46, 711, 0.38]).reshape(1, -1)

worst_neighbor_df = pd.DataFrame(worst_neighbor, columns=array_feature_order)[top6]
best_neighbor_df = pd.DataFrame(best_neighbor, columns=array_feature_order)[top6]

worst_prediction = model_refactored.predict(worst_neighbor_df)[0]
best_prediction = model_refactored.predict(best_neighbor_df)[0]

print(f"Predicción peor escenario: {worst_prediction:.2f}")
print(f"Predicción mejor escenario: {best_prediction:.2f}")

Predicción peor escenario: 3.02
Predicción mejor escenario: 35.31


**Comentario final**

Según el modelo refactorizado, el peor escenario tendría un valor esperado cercano a `3.02`, mientras que el mejor escenario tendría un valor esperado cercano a `35.31`, en las unidades de la variable `medv`.

El modelo completo con intercepto tuvo mejor desempeño predictivo, pero el modelo de 6 atributos permite observar con mayor claridad qué variables están más relacionadas con el valor mediano de las viviendas.